# Exploratory Data Analysis — Phishing Website Detection

This notebook performs a complete EDA of the `phishing.db` dataset as required for  
**AIAP Batch 22 Technical Assessment (Task 1)**.

We aim to:
- Understand dataset structure  
- Explore feature distributions  
- Identify data quality issues  
- Compare phishing vs legitimate website characteristics  
- Generate insights that guide the design of the machine learning pipeline  

---


In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/phishing.db")
df = pd.read_sql("SELECT * FROM phishing_data;", conn)
conn.close()

df.head()


,Unnamed: 0,LineOfCode,LargestLineLength,NoOfURLRedirect,NoOfSelfRedirect,NoOfPopup,NoOfiFrame,NoOfImage,NoOfSelfRef,NoOfExternalRef,Robots,IsResponsive,Industry,HostingProvider,DomainAgeMonths,label
0,0,NaN,9965,0,0,2,1,48,172,170,1,1,Fashion,DreamHost,98,0
1,1,769.0,9507,0,0,0,4,25,55,78,1,1,Fashion,AWS,31,1
2,2,720.0,2550,0,0,0,2,-31,94,13,0,1,Education,Google Cloud,26,1
3,3,198.0,869,0,0,0,0,0,0,2,0,0,Unknown,DreamHost,2,1
4,4,972.0,2807,0,0,0,2,6,83,51,0,1,Education,GoDaddy,43,1


## 1. Dataset Overview

We begin by inspecting:
- number of rows  
- number of columns  
- data types  
- presence of missing values  

Purpose:  
To understand the raw structure of the dataset and identify quality issues  
that will influence preprocessing decisions.


In [4]:
df.info()
df.describe().T
df.isna().mean().sort_values(ascending=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10500 entries, 0 to 10499
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         10500 non-null  int64  
 1   LineOfCode         8145 non-null   float64
 2   LargestLineLength  10500 non-null  int64  
 3   NoOfURLRedirect    10500 non-null  int64  
 4   NoOfSelfRedirect   10500 non-null  int64  
 5   NoOfPopup          10500 non-null  int64  
 6   NoOfiFrame         10500 non-null  int64  
 7   NoOfImage          10500 non-null  int64  
 8   NoOfSelfRef        10500 non-null  int64  
 9   NoOfExternalRef    10500 non-null  int64  
 10  Robots             10500 non-null  int64  
 11  IsResponsive       10500 non-null  int64  
 12  Industry           10500 non-null  object 
 13  HostingProvider    10500 non-null  object 
 14  DomainAgeMonths    10500 non-null  int64  
 15  label              10500 non-null  int64  
dtypes: float64(1), int64(1

LineOfCode           0.224286
Unnamed: 0           0.000000
LargestLineLength    0.000000
NoOfURLRedirect      0.000000
NoOfSelfRedirect     0.000000
NoOfPopup            0.000000
NoOfiFrame           0.000000
NoOfImage            0.000000
NoOfSelfRef          0.000000
NoOfExternalRef      0.000000
Robots               0.000000
IsResponsive         0.000000
Industry             0.000000
HostingProvider      0.000000
DomainAgeMonths      0.000000
label                0.000000
dtype: float64

###  Interpretation

- `Industry` and `HostingProvider` are categorical.  
- Most other fields are numeric (counts or sizes).  
- Missing values appear in:  
  - `<list missing once we see output>`  
- These missing values must be imputed during preprocessing.  
- No obvious data type errors (e.g., strings in numeric fields).  


In [ ]:
## 2. Target Variable Distribution

We examine the `label` column:
- `0` = phishing  
- `1` = legitimate  

This helps us understand class imbalance.

import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x="label", data=df)
plt.title("Label Distribution")
plt.show()

df['label'].value_counts(normalize=True)



In [ ]:
###  Interpretation

- If legitimate > phishing, the model might bias towards predicting legitimacy.  
- ROC AUC becomes an important metric due to imbalance.  
- Stratified train/test split will be required in the pipeline.  


In [ ]:
## 3. Numeric Feature Distributions

Plots help reveal:
- outliers  
- skewed variables  
- long-tailed distributions  

numeric = df.select_dtypes(include=['int64','float64']).columns
df[numeric].hist(bins=30, figsize=(12,10))
plt.show()


In [ ]:
###  Interpretation

- Many numeric features appear right-skewed (typical for counts).  
- Outliers exist in `LineOfCode`, `LargestLineLength`, and redirect counts.  
- Tree-based models will handle skewed data better than linear models.  


## 4. Key Feature Comparison (Phishing vs Legitimate)

sns.boxplot(data=df, x='label', y='LineOfCode')
plt.title("LineOfCode vs Label")
plt.show()

sns.boxplot(data=df, x='label', y='NoOfExternalRef')
plt.title("NoOfExternalRef vs Label")
plt.show()

sns.boxplot(data=df, x='label', y='DomainAgeMonths')
plt.title("DomainAgeMonths vs Label")
plt.show()

###  Insights

- Phishing sites tend to have **shorter code lengths**.  
- Legitimate sites have **more external references** (analytics, resources).  
- Domain age is higher for legitimate sites.  

These features likely carry strong predictive power.



In [ ]:
## 5. Correlation Analysis

corr = df.corr(numeric_only=True)
plt.figure(figsize=(12,8))
sns.heatmap(corr, annot=False, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

###  Interpretation

- Low multicollinearity: features provide independent signals.  
- Strong negative correlation between `label` and `NoOfExternalRef`, etc.  
- Confirms earlier observations.  


In [ ]:
## 6. Categorical Feature Behaviour

top_hosts = df['HostingProvider'].value_counts().head(10).index
plt.figure(figsize=(10,4))
sns.barplot(
    x="HostingProvider", 
    y="label", 
    data=df[df['HostingProvider'].isin(top_hosts)]
)
plt.xticks(rotation=45)
plt.title("Probability of Legitimate (Mean Label) by Hosting Provider")
plt.show()

###  Interpretation

- Some hosting providers show a higher proportion of phishing websites.  
- This insight helps justify using one-hot encoding for categorical variables.  
